# 《PythAPCS123》單元 6-7：未知行數與檔案結尾測資讀取（while 搭配 EOF 處理）

**適合對象**：程式設計初學者（完全零基礎） / APCS 扎根第三十九部曲（第六章大結局第七部曲）  
**對應教材**：吳邦一老師《PythAPCS123：Python 程式設計從 APCS 實作 1 級到 3 級》第 10 章（第 30 頁）、第 17.1 節 EOF 結尾的測資（第 77–78 頁）  

---

### 🌟 本單元學習導航地圖
歡迎各位程式冒險者來到第六章【迴圈結構與控制流程】的最終巔峰——第七座里程碑！

回顧前六個單元，我們所處理的所有題目，測資的規模都是有跡可循的：
- 「*第一行輸入 $n$，代表接下來有 $n$ 筆資料*」（次數已知，`for _ in range(n)`）
- 「*輸入數值直到輸入 0 為止*」（哨兵數值已知，`while True: if x == 0: break`）

然而，在各大程式競賽平台（如 ZeroJudge、Codeforces、UVa / CPE 大學程式能力檢定）中，你一定會頻繁看見以下這種令新手困惑不已的題目描述：  
> **「輸入有多行，每行兩個整數 $a$ 和 $b$，請輸出兩數之和，直到檔案結束（EOF）為止。」**

題目既沒有告訴你總共有幾行（可能有 10 行，也可能有 10 萬行），也沒有在最後一行貼心放個 0 當結束標記！  
面對這種**「未知行數、無預警結束的數據狂流」**，程式如果盲目狂讀，就會在檔案結束的一瞬間被作業系統報錯中斷（崩潰拿到 Runtime Error）；如果不敢讀，就根本無法處理後續測資！

為了解開這個競技程式設計的通關死結，我們將學習 Python 結合作業系統底層的終極絕技——**`while True` 搭配 `try ... except EOFError: break`**！  

本單元精心規劃了五大循序漸進的精華修練關卡：
1. **6-7-1 什麼是 EOF？線上題庫未知筆數測資的真實面貌與鍵盤模擬**
2. **6-7-2 `while True` 搭配 `try ... except EOFError: break` 黃金標準模板**
3. **6-7-3 EOF 模式下的同行多數值讀取：`map()` 與 `split()` 的安全協同**
4. **6-7-4 複合異常捕捉：空白行與格式雜訊的防禦（`except (EOFError, ValueError)`）**
5. **6-7-5 競技程式實戰：未知筆數數據流串流處理（Streaming）大總結**

> 💡 **學習小叮嚀（嚴格零提前依賴）**：本單元**嚴格禁止使用任何串列容器（`list`）、列表生成式、字串切片（`[::-1]`）或自訂函數（`def`）**！所有題目皆在迴圈讀取過程中以常數空間 $O(1)$ 即時完成計算。請跟隨標準五步驟（**說明 ➔ 範例 ➔ 填空 ➔ 練習 ➔ 挑戰**），攜手拿下第六章的最終通關榮耀！


### 🌊 6-7-1 什麼是 EOF？線上題庫未知筆數測資的真實面貌與鍵盤模擬

**生活比喻：自來水管的供水與廣播電台收播**  
想像你家廚房的自來水龍頭。當你把水桶放在水龍頭下方接水時，水流嘩啦嘩啦源源不絕地流進來。你不知道水庫總共會供水多少公升，但你不需要管總量，你只需要「水來了就裝水」；直到某一刻，水庫維修人員把總閥門一關，水龍頭發出一聲吸氣的「嘶～」聲，一滴水也流不出來了！這時候，你就知道「供水徹底結束了，可以收工裝箱了！」  
在電腦檔案系統中，這個「自來水被徹底抽乾、到達檔案最末尾」的特殊信號，就叫做 **EOF（End Of File，檔案結尾）**！

**底層運作機制與 EOFError 的誕生**  
線上評判系統（Online Judge，如 ZeroJudge）在測試你的程式時，並不是由人類在鍵盤前面一個字一個字敲給你，而是**直接把一個預先準備好的文字檔案（測資檔），像倒水一樣一口氣灌進你的程式中（這叫做輸入重定向 Standard Input Redirection）**！  
當文字檔裡面的最後一行被你用 `input()` 讀走之後，檔案已經見底空空如也了。  
如果這時候你的程式還貪心地再次呼叫 `input()` 想要向作業系統要資料，作業系統會無奈地兩手一攤，直接丟出一個嚴重的執行時期異常：  
**`EOFError: EOF when reading a line`（檔案已見底，無資料可讀！）**  
如果你的程式沒有事先準備好防護網，這個 `EOFError` 就會導致程式瞬間暴斃，線上評判系統就會直接亮出紅色的 **RE（Runtime Error）** 判定！

**初學者如何在自己的電腦上「手動觸發 EOF」進行測試？**  
如果你在終端機或 Colab 中執行讀取 EOF 的程式，程式會一直停在畫面上等待輸入，你該怎麼告訴電腦「我輸入完了，這就是檔案結尾」呢？  
- **Windows 終端機環境**：在輸入換行處按下鍵盤組合鍵 **`Ctrl + Z`**，然後按下 **`Enter`** 鍵！  
- **Mac / Linux 終端機環境**：在輸入換行處按下鍵盤組合鍵 **`Ctrl + D`**！  
- **Google Colab 網頁環境**：在 Colab 的輸入框中，若想結束輸入，可直接按下 **`Enter` 鍵送出空行**（若程式有搭配單元 6-7-4 的 `ValueError` 防禦，就會自動跳出中斷）；或者點擊儲存格左側的「停止」按鈕結束！  
作業系統或執行環境收到訊號後，就會立刻觸發結束，讓程式優雅收尾！

**APCS 考試實務建議：**  
吳邦一老師在講義第 78 頁特別提到：APCS 檢定官網過去的題目多半會給定行數或 0 結尾，但 ZeroJudge 與各大學程式解題題庫有海量的題目是 EOF 結尾！掌握 EOF 的防禦技巧，是跨足各大競技解題平台的必經之路。

In [ ]:
# [2] Code 範例區：展示 EOFError 的本質（以文字概念示範）
# 說明：在正常程式中，如果直接無限呼叫 input() 而不防禦，最後必會遭遇 EOFError。
# 本範例展示當遇到 EOFError 時，直譯器回報的錯誤類型名稱。

print("=== EOF（檔案結尾）概念展示 ===")
print("在 Python 中，檔案見底時觸發的專屬例外類型名稱為：EOFError")
print("測試手動發送 EOF 訊號：")
print("- Windows 環境：按下 [Ctrl + Z] 再按 [Enter]")
print("- Mac/Linux 環境：按下 [Ctrl + D]")
print("=== 了解此機制後，請邁向下一節學習 try-except 防護網！===")


In [ ]:
# ==========================================
# [3] Code 填空題
# 任務說明：
# 認識捕捉 EOF 的專屬異常名稱：
# 當讀取未知筆數輸入時，input() 在檔案末尾會引發 EOFError。
# 請在空格處填入正確的異常名稱 EOFError，完成最基礎的捕捉提示：
# ==========================================

error_type = "___"

# 驗證填寫是否為正確的異常關鍵字名稱
if error_type == "EOFError":
    print("恭喜答對！捕捉檔案結尾的專屬異常名稱正是 EOFError！")
else:
    print("請重新檢查，答案應為精確的 EOFError（注意大小寫）！")


In [ ]:
# ==========================================
# [4] Code 練習題
# 任務說明：
# 未知行數打招呼計數器：
# 題目會持續輸入多行字串（每行一個名字），直到遇到 EOF 為止。
# 規則：
# 每讀入一個名字 name，請印出 "Hello, " 加上該名字。
# 檔案結束時（EOF），最後一行印出總共向幾個人打了招呼。
# 提示結構：
# count = 0
# while True:
#     try:
#         name = input()
#     except EOFError:
#         break
#     print("Hello,", name)
#     count += 1
# print("Total:", count)
#
# 【公開測試資料 1】
# 輸入：
# Alice
# Bob
# 預期輸出：
# Hello, Alice
# Hello, Bob
# Total: 2
#
# 【公開測試資料 2】
# 輸入：
# Johnny
# 預期輸出：
# Hello, Johnny
# Total: 1
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：



In [ ]:
# ==========================================
# [5] Code 挑戰題
# 任務說明：
# 未知行數文字長度累加器：
# 持續讀入多行字串，直到遇到 EOF 為止。
# 請統計所有輸入字串的「總字元長度（使用 len(line) 累加）」以及「總行數」（兩數以空格隔開）。
# 例如輸入兩行：
# Cat
# Elephant
# Cat 長度 3，Elephant 長度 8，總長度 11，總行數 2，輸出：11 2。
#
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：



### 🛡️ 6-7-2 `while True` 搭配 `try ... except EOFError: break` 黃金標準模板

**生活比喻：特技空中飛人與安全氣囊網**  
想像馬戲團裡的空中飛人雜技表演。兩位特技演員在高空中飛躍交接（這相當於 `try:` 嘗試讀取下一行資料）。  
在 99% 的情況下，兩人都能完美握手交接，演出順利進行；  
但是在最後謝幕時，拋接員不再伸手了，飛人演員直接凌空墜下——  
這時飛人會摔得粉身碎骨嗎？絕對不會！因為舞台下方早就張開了一張彈性十足的巨大安全氣囊網（這就是 `except EOFError:`）！  
安全網穩穩接住演員，演員笑臉盈盈地站起來向觀眾鞠躬謝幕（`break` 結束表演），整個過程優雅從容，絕不發生意外！

**底層運作機制與語法拆解**  
在 Python 中，處理未知筆數輸入最標準、最乾淨的教科書級黃金語法模板如下：
```python
while True:
    try:
        line = input()      # 步驟一：大膽嘗試讀取下一行！
    except EOFError:        # 步驟二：如果遇到檔案結尾異常（水龍頭關閉）
        break               # 步驟三：立刻優雅跳出無窮迴圈，不再讀取！
    
    # 步驟四：正常處理當前這筆資料
    num = int(line)
    print(num * num)
```

讓我們深入剖析這四個步驟的協同之美：
1. **`while True:`**：因為事前根本不知道有幾行，所以先啟動「永不打烊」的引擎，一直接收資料。
2. **`try:`**：意思是「嘗試執行縮排裡的指令」。如果讀取成功，程式就會順順利利往下走。
3. **`except EOFError:`**：這是專屬的防空洞！一旦 `input()` 抓不到資料拋出 `EOFError`，Python 不會崩潰報錯，而是會**瞬間被吸進 `except` 區塊中**！
4. **`break`**：在防空洞裡踩下煞車，徹底摧毀 `while True` 迴圈，程序順利過渡到迴圈外部！

**常見錯誤陷阱：**
1. **把資料處理寫在 `try` 外面卻沒防護**：一定要先確保 `input()` 成功拿到資料，才能進行後續的運算！
2. **忘記寫 `break`**：若在 `except EOFError:` 肚子裡寫了 `pass` 而不是 `break`，迴圈下一輪又會再去呼叫 `input()`，陷入無限拋出異常的死結！

**APCS 考試實務建議：**  
這是所有競技程式選手走進任何在線評判系統前**必須背得滾瓜爛熟的第一條肌肉記憶指令**！只要看到題目寫「直到 EOF」，大腦請在 0.5 秒內自動敲出 `while True: try: ... except EOFError: break`！

In [ ]:
# [2] Code 範例區：展示單行數值讀取直到 EOF 的標準架構
# 模擬情境：輸入多個整數（每行一個），即時輸出其 10 倍數值

print("=== try-except EOF 讀取架構示範 ===")
print("程式碼核心骨架展示：")
print("while True:")
print("    try:")
print("        n = int(input())")
print("    except EOFError:")
print("        break")
print("    print(n * 10)")
print("====================================")


In [ ]:
# ==========================================
# [3] Code 填空題
# 任務說明：
# 完善 try-except-break 模板：
# 請在空格處依序填入：
# 1. 嘗試執行危險讀取操作的關鍵字（try）。
# 2. 捕捉異常的關鍵字（except）。
# 3. 跳離無窮迴圈的關鍵字（break）。
# ==========================================

# 模擬骨架填空驗證
keyword_1 = "___"  # 應為 try
keyword_2 = "___"  # 應為 except
keyword_3 = "___"  # 應為 break

if keyword_1 == "try" and keyword_2 == "except" and keyword_3 == "break":
    print("太棒了！你已經完全掌握了 EOF 防護三劍客：try ➔ except ➔ break！")
else:
    print("請檢查關鍵字拼寫：依序應為 try, except, break！")


In [ ]:
# ==========================================
# [4] Code 練習題
# 任務說明：
# 未知行數整數總加總器：
# 輸入有多行，每行輸入一個整數，直到 EOF 為止。
# 請計算並輸出所有輸入整數的「累加總和」。
# （若完全無任何輸入即遇到 EOF，總和輸出 0）
#
# 【公開測試資料 1】
# 輸入：
# 10
# 25
# -5
# 預期輸出：
# 30
# （說明：10 + 25 + (-5) = 30）
#
# 【公開測試資料 2】
# 輸入：
# 100
# 200
# 300
# 預期輸出：
# 600
# （說明：100 + 200 + 300 = 600）
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：



In [ ]:
# ==========================================
# [5] Code 挑戰題
# 任務說明：
# 未知行數極值擂台追蹤器：
# 輸入有多行，每行輸入一個整數，直到 EOF 為止（保證至少有一行整數）。
# 請在逐行讀取的過程中即時維護最大值，最終輸出所有輸入整數中的「最大值」。
# 例如輸入四行：
# 15
# 88
# 42
# 73
# 輸出：88。
#
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：



### 📦 6-7-3 EOF 模式下的同行多數值讀取：`map()` 與 `split()` 的安全協同

**生活比喻：流水線上的雙胞胎包裹拆箱**  
想像你在快遞分揀流水線工作。傳送帶上一箱一箱送來包裹，每個箱子打開裡面都整齊裝著兩顆水果（例如蘋果和水梨）。  
你的動作是一氣呵成的：「拿箱子 ➔ 拆箱 ➔ 同時取出兩顆水果（`a, b = map(int, input().split())`）」。  
傳送帶什麼時候停止？沒人知道！只要還有一箱送過來，你就拆一箱；當傳送帶空了（EOF 訊號傳來），伸手抓空時，你立刻轉身下班！  
這就是線上題庫最熱門、出題頻率最高的題目型態——**同行多數值讀取直到 EOF**！

**底層運作機制與語法結合**  
在第四章 4.6 節中，我們學會了單行讀取兩個或多個整數的標準語法：  
`a, b = map(int, input().split())`  
當我們把這個強大的一行讀取語法，安放進 `try ... except EOFError:` 的溫暖懷抱時，神奇的化學反應發生了：
```python
while True:
    try:
        # 一行搞定：讀取整行 ➔ 空白切分 ➔ 轉型為整數 ➔ 同時賦予 a 與 b
        a, b = map(int, input().split())
    except EOFError:
        break  # 沒有下一行了，乾淨跳出！
        
    # 每一行讀入後立即運算並輸出
    print(a + b)
```
這段短短 7 行的程式碼，就是經典題 **ZeroJudge a002（簡易加法直到 EOF）** 的滿分標準解答！

**初學者最容易掉入的兩大致命陷阱：**
1. **陷阱一：變數個數與輸入數字個數不匹配**  
   如果題目每行只給 2 個數字，左側卻寫了 `a, b, c = map(...)`，會引發 `ValueError: not enough values to unpack`！解題前務必看清題目每行有幾個數字。
2. **陷阱二：誤以為要把所有答案存起來最後一次印**  
   很多初學同學誤以為「必須等所有測資都讀完，才能一次全部印出答案」，於是大費周章想找容器存答案。  
   請牢牢記住競技解題的核心法則：**「裁判看的是標準輸出資料流（stdout）！每讀一行、算出一行、立刻 print 一行，完全合法，且記憶體消耗永遠是最小的常數級！」**

**APCS 考試實務建議：**  
「讀一筆、算一筆、印一筆」是競賽解題中最優美的流式處理（Streaming）哲學，不僅程式碼極短，而且永遠不用擔心數百萬筆資料會把電腦記憶體撐爆！

In [ ]:
# [2] Code 範例區：展示兩數讀取並即時求差值的 EOF 架構
# 經典題型：每行輸入兩個整數 a, b，輸出兩數的絕對差值 |a - b|

print("=== 兩數讀取即時輸出架構展示 ===")
print("while True:")
print("    try:")
print("        a, b = map(int, input().split())")
print("    except EOFError:")
print("        break")
print("    print(abs(a - b))")
print("================================")


In [ ]:
# ==========================================
# [3] Code 填空題
# 任務說明：
# 補齊同行雙數讀取的黃金語法：
# 每行輸入兩個以空格隔開的整數時，最經典的拆解語法為：
# a, b = map(int, input().split())
# 請將下方字串中的 ___ 替換為正確的函式名稱：
# 1. 切分字串的函式名稱（應填 "split"）
# 2. 將字串元素轉換為整數型態的函式名稱（應填 "int"）
# ==========================================

func_split = "___"  # 1. 切分字串的函式名稱（填入 "split"）
func_type = "___"   # 2. 轉換型態的函式名稱（填入 "int"）

if func_split == "split" and func_type == "int":
    print("🎉 答對了！map(int, input().split()) 是拆解同行多數值的黃金拍檔！")
else:
    print("💡 請重新檢查：func_split 應為 \"split\"，func_type 應為 \"int\"！")


In [ ]:
# ==========================================
# [4] Code 練習題
# 任務說明：
# 兩數較大者探測器（改編自經典題 ZeroJudge a002 變形）：
# 輸入包含多行，每行有兩個以空格隔開的整數 a 與 b，直到 EOF 為止。
# 規則：
# 每讀入一行，請輸出該行中較大的那個數字（若兩數相等，則印出該數值）。
#
# 【公開測試資料 1】
# 輸入：
# 5 8
# 12 3
# 預期輸出：
# 8
# 12
#
# 【公開測試資料 2】
# 輸入：
# -10 -20
# 7 7
# 預期輸出：
# -10
# 7
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：



In [ ]:
# ==========================================
# [5] Code 挑戰題
# 任務說明：
# 歐幾里得連續大作戰（結合單元 6-4 的最大公因數 GCD）：
# 輸入包含多行，每行有兩個正整數 x 與 y（以空格隔開），直到 EOF 為止。
# 規則：
# 請對每一行的 x 與 y，使用輾轉相除法計算其「最大公因數 GCD」，並印出結果。
# 例如輸入兩行：
# 12 18
# 35 14
# 輸出：
# 6
# 7
#
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：



### 🛡️ 6-7-4 複合異常捕捉：空白行與格式雜訊的防禦（`except (EOFError, ValueError)`）

**生活比喻：快遞包裹裡的空盒子與瑕疵品**  
想像快遞公司運送包裹時，除了正常的包裹，偶爾還會混進幾種令人頭痛的「突發狀況」：  
1. 運送帶空了，完全沒有包裹了（`EOFError` 檔案結束）。  
2. 運送帶上送來了一個「空紙盒」，裡面什麼都沒有（文件最後多餘的空白行或空行！）。  
3. 紙盒裡原本應該裝 2 個零件，結果工人偷懶只裝了 1 個零件，或是塞了一張廢紙（資料格式不合規 `ValueError`）！  
身為一個有經驗的品檢員，你不可能遇到空盒子就當場嚇得暈倒報警；你會淡定地把它們一併歸類為「無效雜訊」，優雅地結束今天的工作！

**底層運作機制與多重異常捕捉**  
在某些線上評判系統（如 UVa Online Judge 或 ZeroJudge 的部分題目）中，題目的測資檔案最後面常常會不小心多留了一行「空行（沒有任何字元，只有一個換行符號 `\n`）」，或者使用者不小心按了幾次回車鍵。  
如果我們只寫 `except EOFError:`，當程式讀到空行時：
- `input().split()` 會得到一個空清單 `[]`！
- 接著 `a, b = map(...)` 發現根本沒有東西可以拆箱，就會當場爆出：  
  **`ValueError: not enough values to unpack (expected 2, got 0)`**！  
此時因為你的 `except` 只防守了 `EOFError`，沒有防守到 `ValueError`，程式依然會在最後一瞬間崩潰拿到 RE！

**兩大終極防禦術：**
1. **寫法一：複合異常捕捉（精確防守）**：  
   用小括號把可能發生的異常打包在一起：
   ```python
   while True:
       try:
           a, b = map(int, input().split())
       except (EOFError, ValueError):  # 同時防禦檔案見底與空行雜訊！
           break
       print(a + b)
   ```
2. **寫法二：吳邦一老師講義傳授的終極無差別防護（全能護盾）**：  
   吳老師在講義第 78 頁特別指出：「*這裡因為只會有 EOF 或格式錯誤，所以寫不寫 EOFError 都可以！*」  
   直接寫 `except:` 或 `except Exception:`：
   ```python
   while True:
       try:
           a, b = map(int, input().split())
       except:  # 只要碰上任何無法讀取的意外，一律安全下車！
           break
       print(a + b)
   ```

**APCS 考試實務建議：**  
遇到題庫測資不乾淨、容易出現多餘空白行時，使用 `except:` 或 `except (EOFError, ValueError): break` 能為你的程式加上最堅固的「防彈玻璃」，徹底阻絕任何偶發性的格式崩潰！

In [ ]:
# [2] Code 範例區：展示全能防禦的 EOF 讀取架構

print("=== 全能防禦架構展示 ===")
print("while True:")
print("    try:")
print("        a, b = map(int, input().split())")
print("    except (EOFError, ValueError):  # 雙重防禦！")
print("        break")
print("    print(a + b)")
print("========================")


In [ ]:
# ==========================================
# [3] Code 填空題
# 任務說明：
# 複合異常防護網：
# 為了同時防禦「檔案見底（EOFError）」與「空行雜訊或解析失敗（ValueError）」，
# 我們可以在 except 後方使用小括號將多個異常名稱打包。
# 請將下方字串中的 ___ 替換為 "(EOFError, ValueError)"：
# ==========================================

defense_syntax = "___"  # 請填入 "(EOFError, ValueError)"

if defense_syntax == "(EOFError, ValueError)":
    print("🎉 答對了！使用 (EOFError, ValueError) 能打造全方位的輸入防護網！")
else:
    print("💡 請重新檢查：defense_syntax 應填為 \"(EOFError, ValueError)\"（注意小括號與逗號空格）！")


In [ ]:
# ==========================================
# [4] Code 練習題
# 任務說明：
# 及格學生人數統計器（健壯性防禦實戰）：
# 讀入多行分數（每行一個整數 score），直到檔案結束或遇到空行中斷。
# 規則：
# 若分數 score >= 60，及格人數 pass_count 加 1。
# 結束後，輸出及格的總人數。
#
# 【公開測試資料 1】
# 輸入：
# 85
# 40
# 60
# 92
# 預期輸出：
# 3
# （說明：85, 60, 92 及格，共 3 人）
#
# 【公開測試資料 2】
# 輸入：
# 59
# 45
# 預期輸出：
# 0
# （說明：無人及格，輸出 0）
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：



In [ ]:
# ==========================================
# [5] Code 挑戰題
# 任務說明：
# 三角形合法性連續判別（直到 EOF）：
# 輸入有多行，每行包含三個正整數 a, b, c（代表三角形三邊長），直到 EOF 結束。
# 規則：
# 若三邊能構成合法三角形（任兩邊之和大於第三邊：a+b>c and a+c>b and b+c>a），
# 印出 YES；否則印出 NO。
# 例如輸入：
# 3 4 5
# 1 2 3
# 輸出：
# YES
# NO
#
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：



### 🏆 6-7-5 競技程式實戰：未知筆數數據流串流處理（Streaming）大總結

**生活比喻：城市自來水廠的總監控中心**  
想像你坐在自來水廠的總監控螢幕前。全市數萬條水管的水流日夜不停地注入蓄水池。  
身為水廠總工程師，你的職責不是把「過去一年每一秒鐘的水滴都做成標本裝進瓶子保存起來（那會把整個倉庫撐爆！）」；  
你的儀表板上只有四個即時指針：  
1. **總水量計數器（累加器 `total_volume`）**  
2. **供水次數計數器（`total_batches`）**  
3. **歷史最高水壓峰值（`max_pressure`）**  
4. **歷史最低水壓谷值（`min_pressure`）**  
每注入一筆水，指針就微調一次；當供水全部停止的那一刻，四個指針上的數字就是這場戰役最完美的總結報表！  
這就是演算法世界中最極致的架構境界——**常數空間 $O(1)$ 串流處理（Streaming Processing）**！

**第六章全武藝大融合：四大法寶協同作戰**  
當我們把第六章所學的所有精華合為一體時，我們會發現：面對無限長度的未知資料流，我們不需要任何複雜的高級容器，就能解決一切統計任務：
1. **`while True + try-except`**（單元 6-7）：負責穩定提供無盡的資料流，檔案結束安全剎車。
2. **累加器 `total += val`**（單元 6-2）：負責實時吞吐總量。
3. **計數器 `count += 1`**（單元 6-2）：負責精準統計總筆數。
4. **擂台盟主動態極值**（單元 6-2）：第一筆即位，後續挑戰篡位，實時捕捉全局最大與最小值。
5. **`continue` 與 `break`**（單元 6-3）：遇到雜訊跳過，遇到熔斷立刻結束！

**APCS 考試實務總結：**  
恭喜各位冒險者！走完單元 6-7，你已經徹底征服了程式設計中最重要的基礎核心——迴圈結構。從有界的等差數列，到未知的條件收斂，再到無限的檔案資料流，所有的迴圈控制力都已化為你的本能！

In [ ]:
# [2] Code 範例區：EOF 串流大數據實時統計示範（概念展示）

print("=== EOF 數據流串流處理模板 ===")
print("count = 0")
print("total = 0")
print("while True:")
print("    try:")
print("        x = int(input())")
print("    except (EOFError, ValueError):")
print("        break")
print("    total += x")
print("    count += 1")
print("if count > 0:")
print("    print('總和：', total, '平均：', total // count)")
print("==============================")


In [ ]:
# ==========================================
# [3] Code 填空題
# 任務說明：
# 串流平均值計算器骨架：
# 請在空格處填入：
# 1. 累加器累加數值的複合賦值運算子（+=）。
# 2. 計數器每次加 1 的複合賦值運算子（+=）。
# ==========================================

total = 0
count = 0

# 模擬 3 筆資料 10, 20, 30 的累加過程（使用 range 產生）
for x in range(10, 40, 10):
    total ___ x      # 1. 填入累加運算子（+=）
    count ___ 1      # 2. 填入計數運算子（+=）

print("總和：", total, "筆數：", count)


In [ ]:
# ==========================================
# [4] Code 練習題
# 任務說明：
# 未知行數全距計算器（Range Calculator）：
# 輸入包含多行，每行一個整數，直到 EOF 為止（保證至少有 1 筆整數）。
# 規則：
# 請在走訪過程中即時維護最大值 max_val 與最小值 min_val，
# 檔案結束時，輸出所有整數的「全距」（即 最大值 - 最小值）。
# 提示：第一筆資料可直接作為最初的 max_val 與 min_val。
#
# 【公開測試資料 1】
# 輸入：
# 15
# 8
# 22
# 10
# 預期輸出：
# 14
# （說明：最大值 22，最小值 8，全距為 22 - 8 = 14）
#
# 【公開測試資料 2】
# 輸入：
# 50
# 50
# 預期輸出：
# 0
# （說明：最大值 50，最小值 50，全距為 0）
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：



In [ ]:
# ==========================================
# [5] Code 挑戰題
# 任務說明：
# 數據流合格率與平均值大結算：
# 輸入包含多行，每行輸入一個學生考試成績（0~100 的整數），直到 EOF 結束。
# 規則：
# 1. 統計所有學生的總人數 total_students。
# 2. 統計及格（score >= 60）的學生人數 pass_students。
# 3. 計算全班成績的總和，並計算全班平均分數（使用整數除法 //）。
# 輸出說明：
# 請輸出全班總人數、及格人數、全班平均分數（三數以空格隔開）。
# （若資料完全為空，請輸出 0 0 0）
# 例如輸入：
# 80
# 40
# 90
# 總人數 3，及格人數 2，平均 (80+40+90)//3 = 70，輸出：3 2 70。
#
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：



---
### 🏆 恭喜完成單元 6-7，榮獲「數據串流宗師」至尊徽章！
### 🎊 賀！第六章【迴圈結構與控制流程】全數大圓滿完工！

太不可思議了！在單元 6-7 中，各位自學冒險者成功解鎖了競技程式設計的通關金鑰：
1. **透徹理解 EOF 本質**：明白檔案結尾在作業系統與在線題庫（OJ）中的運作真相與鍵盤模擬方式（`Ctrl+Z` / `Ctrl+D`）！
2. **try-except-break 黃金模板**：建立防彈級的輸入讀取架構，從此徹底告別令人聞風喪膽的 `EOFError` 與 `Runtime Error`！
3. **同行多數值串流讀取**：掌握 `map(int, input().split())` 結合 EOF 的「讀一筆、算一筆、印一筆」極速流式處理！
4. **全能異常雙重防禦**：使用 `except (EOFError, ValueError): break`，優雅過濾題庫多餘空行與雜訊！
5. **常數空間 $O(1)$ 串流大數據分析**：以最精簡純粹的變數，即時搞定未知巨量資料的加總、計數與極值維護！

---

### 🗺️ 第六章學習進度全覽（7 / 7 全數攻克完工！）：
- [x] **6.1 計數迴圈（for 搭配 range）**（range 單雙三參數、左閉右開與負步進倒數）
- [x] **6.2 迴圈累加器與計數器模式**（歸零初始化、逐回合數值累加、條件計數與擂台求極值）
- [x] **6.3 迴圈中斷與跳步（break, continue）**（提早破圈而出與略過本回合剩餘指令）
- [x] **6.4 條件迴圈（while 迴圈）**（未知次數迭代、輾轉相除法與位數逐位拆解）
- [x] **6.5 雙重與多重巢狀迴圈**（外層帶動內層、九九乘法表排版與幾何星號圖形）
- [x] **6.6 迴圈控制變數的常見陷阱**（走訪中竄改變數無效、倒數邊界失誤與無窮迴圈）
- [x] **6.7 未知行數與檔案結尾測資讀取（while 搭配 EOF 處理）**（競技程式未知筆數讀取絕技）

---

👉 **下章預告：【第七章 字串（String）特性與序列操作】**  
告別了純數值的世界，我們即將跨入美麗多彩的「文字魔法王國」！  
電腦是如何用一個個字元拼出長長的單字與文章？  
文字的秘密編碼、字串的相加與重複、反斜線跳脫字元 `\n`，以及無比強大的「字串切片與反轉判斷迴文」！  
第七章的大門即將為你開啟，讓我們繼續勇往直前！
